# AlphaGenome Gene Expression Predictions — Evaluation

This notebook evaluates the predicted human/chimp gene expression LFC values produced by `ag_predictions.py`.

**Sections**
1. Setup & Data Loading
2. Data Quality Overview
3. LFC Distribution
4. Top Differentially Expressed Genes
5. Species Expression Levels
6. Comparison with Hybrid ASE Data (LFC correlation & direction accuracy)
7. Predicted vs Experimental Per-Species Expression
8. Lineage-Causing Change Analysis
9. Summary

## 1. Setup & Data Loading

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import pearsonr, spearmanr, gaussian_kde
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import config

RESULTS_DIR = Path('./results/all_genes')
FIGSIZE_WIDE = (14, 5)
FIGSIZE_SQ   = (7, 6)

In [ ]:
tsv_files = sorted(RESULTS_DIR.glob('lfc_df_*.tsv'))
print(f'Found {len(tsv_files)} result chunk(s)')

if len(tsv_files) == 0:
    raise FileNotFoundError(f'No result TSVs found in {RESULTS_DIR}. Wait for cluster jobs to finish.')

df = pl.concat([pl.read_csv(f, separator='\t') for f in tsv_files])
print(f'Total rows loaded : {df.height:,}')
print(f'Columns           : {df.columns}')
df.head()

## 2. Data Quality Overview

In [ ]:
total        = df.height
null_lfc     = df['LFC'].null_count()
null_human   = df['HumanGeneExpression'].null_count()
null_chimp   = df['ChimpGeneExpression'].null_count()
complete     = df.drop_nulls(subset=['LFC', 'HumanGeneExpression', 'ChimpGeneExpression']).height

print(f'Total gene pairs processed : {total:,}')
print(f'Complete pairs (both expr) : {complete:,}  ({100*complete/total:.1f}%)')
print(f'Missing LFC                : {null_lfc:,}')
print(f'Missing human expression   : {null_human:,}')
print(f'Missing chimp expression   : {null_chimp:,}')

In [ ]:
# Breakdown: both missing vs one species missing
missing_human_only  = df.filter(df['HumanGeneExpression'].is_null() & df['ChimpGeneExpression'].is_not_null()).height
missing_chimp_only  = df.filter(df['ChimpGeneExpression'].is_null() & df['HumanGeneExpression'].is_not_null()).height
missing_both        = df.filter(df['HumanGeneExpression'].is_null() & df['ChimpGeneExpression'].is_null()).height

labels  = ['Complete', 'Missing human only', 'Missing chimp only', 'Missing both']
sizes   = [complete, missing_human_only, missing_chimp_only, missing_both]
colors  = ['#4CAF50', '#FF9800', '#2196F3', '#F44336']

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    sizes, labels=labels, colors=colors,
    autopct=lambda p: f'{p:.1f}%' if p > 1 else '',
    startangle=140
)
ax.set_title('Gene Pair Completion Status')
plt.tight_layout()
plt.show()

for l, s in zip(labels, sizes):
    print(f'  {l:<25}: {s:,}')

In [ ]:
# Work with complete pairs from here on
df_clean = df.drop_nulls(subset=['LFC', 'HumanGeneExpression', 'ChimpGeneExpression'])
lfc        = df_clean['LFC'].to_numpy()
human_expr = df_clean['HumanGeneExpression'].to_numpy()
chimp_expr = df_clean['ChimpGeneExpression'].to_numpy()
print(f'Working with {len(lfc):,} complete gene pairs')

## 3. LFC Distribution

In [ ]:
mean_lfc   = np.mean(lfc)
median_lfc = np.median(lfc)
std_lfc    = np.std(lfc)
n_human_biased = np.sum(lfc > 0)
n_chimp_biased = np.sum(lfc < 0)

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE_WIDE)

# Histogram
ax = axes[0]
ax.hist(lfc, bins=80, color='steelblue', alpha=0.75, edgecolor='none')
ax.axvline(mean_lfc,   color='red',    linestyle='--', linewidth=1.5, label=f'Mean = {mean_lfc:.2f}')
ax.axvline(median_lfc, color='orange', linestyle=':',  linewidth=1.5, label=f'Median = {median_lfc:.2f}')
ax.axvline(0,          color='black',  linestyle='-',  linewidth=1,   alpha=0.4)
ax.set_xlabel('log\u2082 Fold Change (Human / Chimp)')
ax.set_ylabel('Gene count')
ax.set_title('LFC Distribution')
ax.legend()
stats_text = f'N = {len(lfc):,}\nMean = {mean_lfc:.2f}\nMedian = {median_lfc:.2f}\nSD = {std_lfc:.2f}'
ax.text(0.97, 0.97, stats_text, transform=ax.transAxes, va='top', ha='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Cumulative
ax = axes[1]
sorted_lfc = np.sort(lfc)
cumulative = np.arange(1, len(sorted_lfc)+1) / len(sorted_lfc)
ax.plot(sorted_lfc, cumulative, color='steelblue', linewidth=1.5)
ax.axvline(0,   color='black',  linestyle='--', linewidth=1, alpha=0.5)
ax.axhline(0.5, color='orange', linestyle=':',  linewidth=1)
ax.set_xlabel('log\u2082 Fold Change (Human / Chimp)')
ax.set_ylabel('Cumulative fraction')
ax.set_title('Cumulative LFC Distribution')

plt.tight_layout()
plt.show()

print(f'Human-biased genes (LFC > 0): {n_human_biased:,}  ({100*n_human_biased/len(lfc):.1f}%)')
print(f'Chimp-biased genes  (LFC < 0): {n_chimp_biased:,}  ({100*n_chimp_biased/len(lfc):.1f}%)')

In [ ]:
# LFC magnitude breakdown
thresholds = [1, 2, 3, 5]
print('LFC magnitude breakdown (|LFC| threshold):')
print(f'  {"Threshold":<12} {"Human-biased":>14} {"Chimp-biased":>14}')
for t in thresholds:
    nh = np.sum(lfc >  t)
    nc = np.sum(lfc < -t)
    print(f'  |LFC| > {t:<4}  {nh:>10,} ({100*nh/len(lfc):5.1f}%)  {nc:>10,} ({100*nc/len(lfc):5.1f}%)')

## 4. Top Differentially Expressed Genes

In [ ]:
TOP_N = 20

top_human = df_clean.sort('LFC', descending=True).head(TOP_N).select(
    ['GeneSymbol', 'HumanGeneID', 'ChimpGeneID', 'LFC', 'HumanGeneExpression', 'ChimpGeneExpression']
)
top_chimp = df_clean.sort('LFC', descending=False).head(TOP_N).select(
    ['GeneSymbol', 'HumanGeneID', 'ChimpGeneID', 'LFC', 'HumanGeneExpression', 'ChimpGeneExpression']
)

print(f'=== Top {TOP_N} Human-upregulated genes ===')
print(top_human.to_pandas().to_string(index=False))
print()
print(f'=== Top {TOP_N} Chimp-upregulated genes ===')
print(top_chimp.to_pandas().to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, data, title, color in [
    (axes[0], top_human, f'Top {TOP_N} Human-upregulated', 'steelblue'),
    (axes[1], top_chimp, f'Top {TOP_N} Chimp-upregulated', 'tomato'),
]:
    genes = data['GeneSymbol'].to_list()
    lfcs  = data['LFC'].to_numpy()
    ax.barh(range(len(genes)), lfcs, color=color, alpha=0.8)
    ax.set_yticks(range(len(genes)))
    ax.set_yticklabels(genes, fontsize=9)
    ax.invert_yaxis()
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('log\u2082 Fold Change (Human / Chimp)')
    ax.set_title(title)

plt.tight_layout()
plt.show()

## 5. Species Expression Levels

In [ ]:
log_human = np.log10(human_expr + 1e-9)
log_chimp = np.log10(chimp_expr + 1e-9)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution per species
ax = axes[0]
ax.hist(log_human, bins=60, alpha=0.6, label='Human', color='steelblue')
ax.hist(log_chimp, bins=60, alpha=0.6, label='Chimp', color='tomato')
ax.set_xlabel('log\u2081\u2080(predicted expression)')
ax.set_ylabel('Gene count')
ax.set_title('Predicted Expression Distributions')
ax.legend()

# Human vs chimp scatter (density-coloured)
ax = axes[1]
mask = np.isfinite(log_human) & np.isfinite(log_chimp)
xp, yp = log_human[mask], log_chimp[mask]
xy = np.vstack([xp, yp])
z  = gaussian_kde(xy)(xy)
idx = z.argsort()
sc = ax.scatter(xp[idx], yp[idx], c=z[idx], cmap='viridis', s=8, alpha=0.7)
lims = [min(xp.min(), yp.min()) - 0.2, max(xp.max(), yp.max()) + 0.2]
ax.plot(lims, lims, 'r--', linewidth=1.2, label='x = y')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('log\u2081\u2080(human predicted expression)')
ax.set_ylabel('log\u2081\u2080(chimp predicted expression)')
ax.set_title('Human vs Chimp Expression')
r, p = pearsonr(xp, yp)
ax.text(0.05, 0.95, f'r = {r:.3f}\np = {p:.2e}', transform=ax.transAxes,
        va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
plt.colorbar(sc, ax=ax, label='Density')
ax.legend()

# Expression vs |LFC|
ax = axes[2]
mean_log_expr = (log_human + log_chimp) / 2
abs_lfc = np.abs(lfc)
mask2 = np.isfinite(mean_log_expr) & np.isfinite(abs_lfc)
ax.scatter(mean_log_expr[mask2], abs_lfc[mask2], s=4, alpha=0.3, color='purple')
ax.set_xlabel('Mean log\u2081\u2080(expression)')
ax.set_ylabel('|LFC|')
ax.set_title('Expression Level vs |LFC|')
r2, p2 = pearsonr(mean_log_expr[mask2], abs_lfc[mask2])
ax.text(0.05, 0.95, f'r = {r2:.3f}\np = {p2:.2e}', transform=ax.transAxes,
        va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

plt.tight_layout()
plt.show()

## 6. Comparison with Hybrid ASE Data

In [ ]:
hybrids = pl.read_csv(config.HUMAN_CHIMP_HYBRIDS_DATA_PATH_WEXAC, separator='\t')
print(f'Hybrid ASE rows : {hybrids.height:,}')
print(f'Columns         : {hybrids.columns}')
hybrids.head(3)

In [ ]:
# Note: prediction output uses 'GeneSymbol'; hybrid data uses 'Gene'
joined = df_clean.join(hybrids, left_on='GeneSymbol', right_on='Gene', how='inner')
print(f'Genes matched to hybrid ASE data: {joined.height:,}')
joined.head(3)

In [ ]:
# --- 6a. Predicted LFC vs Experimental LFC ---
pred_lfc = joined['LFC'].to_numpy()
exp_lfc  = joined['ExpLBM_LFC_human_ref'].to_numpy()

mask = np.isfinite(pred_lfc) & np.isfinite(exp_lfc)
x, y = pred_lfc[mask], exp_lfc[mask]

xy = np.vstack([x, y])
z  = gaussian_kde(xy)(xy)
idx = z.argsort()

r_p, p_p = pearsonr(x, y)
r_s, p_s = spearmanr(x, y)
m, b = np.polyfit(x, y, 1)

fig, ax = plt.subplots(figsize=FIGSIZE_SQ)
sc = ax.scatter(x[idx], y[idx], c=z[idx], cmap='viridis', s=15, alpha=0.8)
x_line = np.linspace(x.min(), x.max(), 200)
ax.plot(x_line, m*x_line + b, color='red', linewidth=1.5, label='Regression')
ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
ax.axvline(0, color='grey', linewidth=0.8, linestyle='--')
ax.set_xlabel('Predicted log\u2082 FC (AlphaGenome)')
ax.set_ylabel('Experimental log\u2082 FC (Hybrid ASE)')
ax.set_title('Predicted vs Experimental LFC')
plt.colorbar(sc, ax=ax, label='Point density')
stats_text = f'N = {len(x):,}\nPearson r = {r_p:.3f}  p = {p_p:.2e}\nSpearman r = {r_s:.3f}  p = {p_s:.2e}'
ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- 6b. log10-normalised scatter ---
log_pred = np.log10(np.abs(pred_lfc) + 1e-6) * np.sign(pred_lfc)
log_exp  = np.log10(np.abs(exp_lfc)  + 1e-6) * np.sign(exp_lfc)

mask = np.isfinite(log_pred) & np.isfinite(log_exp)
xn = log_pred[mask] - np.mean(log_pred[mask])
yn = log_exp[mask]  - np.mean(log_exp[mask])

xy = np.vstack([xn, yn])
z  = gaussian_kde(xy)(xy)
idx = z.argsort()

r_norm, p_norm = pearsonr(xn, yn)

fig, ax = plt.subplots(figsize=FIGSIZE_SQ)
sc = ax.scatter(xn[idx], yn[idx], c=z[idx], cmap='viridis', s=15, alpha=0.8)
lims = [min(xn.min(), yn.min())-0.1, max(xn.max(), yn.max())+0.1]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='x = y')
ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
ax.axvline(0, color='grey', linewidth=0.8, linestyle='--')
ax.set_xlabel('Predicted LFC (log10-scaled, mean-centred)')
ax.set_ylabel('Experimental LFC (log10-scaled, mean-centred)')
ax.set_title('Predicted vs Experimental LFC (log10-normalised)')
plt.colorbar(sc, ax=ax, label='Point density')
ax.text(0.05, 0.95, f'N = {len(xn):,}\nr = {r_norm:.3f}\np = {p_norm:.2e}',
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- 6c. Direction accuracy ---
direction_df = joined.with_columns(
    pl.col('LFC').sign().alias('pred_direction'),
    pl.col('ExpLBM_LFC_human_ref').sign().alias('exp_direction'),
).drop_nulls(subset=['pred_direction', 'exp_direction']).filter(
    (pl.col('pred_direction') != 0) & (pl.col('exp_direction') != 0)
)

n_agree     = direction_df.filter(pl.col('pred_direction') == pl.col('exp_direction')).height
n_disagree  = direction_df.filter(pl.col('pred_direction') != pl.col('exp_direction')).height
n_dir_total = direction_df.height

r_dir, p_dir = pearsonr(
    direction_df['pred_direction'].to_numpy(),
    direction_df['exp_direction'].to_numpy()
)

print(f'Direction agreement : {n_agree:,} / {n_dir_total:,}  ({100*n_agree/n_dir_total:.1f}%)')
print(f'Pearson r (direction) = {r_dir:.3f}  p = {p_dir:.2e}')

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie([n_agree, n_disagree],
       labels=['Agree\n(same direction)', 'Disagree\n(opposite direction)'],
       colors=['#4CAF50', '#F44336'],
       autopct='%1.1f%%', startangle=90)
ax.set_title(f'Direction Agreement  (N={n_dir_total:,})')
plt.tight_layout()
plt.show()

In [ ]:
# --- 6d. Confusion matrix: human-biased / chimp-biased classification ---
from sklearn.metrics import confusion_matrix, classification_report

pred_cls = (direction_df['pred_direction'].to_numpy() > 0).astype(int)  # 1=human, 0=chimp
exp_cls  = (direction_df['exp_direction'].to_numpy()  > 0).astype(int)

cm = confusion_matrix(exp_cls, pred_cls)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_xticklabels(['Pred: Chimp', 'Pred: Human'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['Exp: Chimp', 'Exp: Human'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=14)
ax.set_title('Confusion Matrix — Human vs Chimp Bias')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print(classification_report(exp_cls, pred_cls, target_names=['Chimp-biased', 'Human-biased']))

## 7. Predicted vs Experimental Per-Species Expression

In [ ]:
def scatter_pred_vs_exp(ax, exp_raw, pred_raw, xlabel, ylabel, title):
    x_raw = np.log10(np.array(exp_raw,  dtype=float))
    y_raw = np.log10(np.array(pred_raw, dtype=float))
    mask = np.isfinite(x_raw) & np.isfinite(y_raw)
    x = x_raw[mask] - np.mean(x_raw[mask])
    y = y_raw[mask] - np.mean(y_raw[mask])
    xy = np.vstack([x, y])
    z  = gaussian_kde(xy)(xy)
    idx = z.argsort()
    r, p = pearsonr(x, y)
    sc = ax.scatter(x[idx], y[idx], c=z[idx], cmap='viridis', s=10, alpha=0.7)
    lims = [min(x.min(), y.min())-0.1, max(x.max(), y.max())+0.1]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='x = y')
    ax.set_xlim(lims); ax.set_ylim(lims); ax.set_aspect('equal', adjustable='box')
    ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
    ax.axvline(0, color='grey', linewidth=0.8, linestyle='--')
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(title)
    ax.text(0.05, 0.95, f'N = {len(x):,}\nr = {r:.3f}\np = {p:.2e}',
            transform=ax.transAxes, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    return sc

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc1 = scatter_pred_vs_exp(
    axes[0],
    joined['ExpLBM_TPM_human_allele'].to_numpy(),
    joined['HumanGeneExpression'].to_numpy(),
    'Experimental human expression (log10 TPM, centred)',
    'Predicted human expression (log10, centred)',
    'Human: Predicted vs Experimental'
)
sc2 = scatter_pred_vs_exp(
    axes[1],
    joined['ExpLBM_TPM_chimp_allele'].to_numpy(),
    joined['ChimpGeneExpression'].to_numpy(),
    'Experimental chimp expression (log10 TPM, centred)',
    'Predicted chimp expression (log10, centred)',
    'Chimp: Predicted vs Experimental'
)

for ax, sc in [(axes[0], sc1), (axes[1], sc2)]:
    plt.colorbar(sc, ax=ax, label='Point density')
    ax.legend()

plt.tight_layout()
plt.show()

## 8. Lineage-Causing Change Analysis

In [ ]:
lineage = pl.read_csv(config.LINEAGE_CAUSING_CHANGE_DATA, separator='\t')
print(f'Lineage data rows : {lineage.height:,}')
print(f'Columns           : {lineage.columns}')
lineage.head(3)

In [ ]:
# Join predictions with lineage data — adjust key column name if needed
gene_col_lineage = [c for c in lineage.columns if 'gene' in c.lower() or 'symbol' in c.lower()]
print(f'Candidate gene columns in lineage data: {gene_col_lineage}')

if gene_col_lineage:
    key = gene_col_lineage[0]
    lineage_joined = df_clean.join(lineage, left_on='GeneSymbol', right_on=key, how='inner')
    print(f'Matched genes: {lineage_joined.height:,}')
    lineage_joined.head(3)
else:
    print('Could not auto-detect gene column — adjust join key manually.')

In [ ]:
# Plot predicted LFC stratified by lineage-causing change category
cat_cols = [c for c in lineage_joined.columns if 'lineage' in c.lower() or 'category' in c.lower() or 'polariz' in c.lower()]
print(f'Candidate category columns: {cat_cols}')

if cat_cols:
    cat_col = cat_cols[0]
    categories = lineage_joined[cat_col].unique().to_list()
    print(f'Using: {cat_col}  |  categories: {categories}')

    fig, ax = plt.subplots(figsize=(8, 5))
    data_by_cat = [
        lineage_joined.filter(pl.col(cat_col) == c)['LFC'].drop_nulls().to_numpy()
        for c in categories
    ]
    ax.boxplot(data_by_cat, labels=categories, patch_artist=True)
    ax.axhline(0, color='red', linestyle='--', linewidth=1)
    ax.set_xlabel('Lineage category')
    ax.set_ylabel('Predicted log\u2082 FC (Human / Chimp)')
    ax.set_title('Predicted LFC by Lineage-Causing Change Category')
    plt.xticks(rotation=15, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No category column found — inspect lineage_joined.columns and adjust.')

## 9. Summary

In [ ]:
print('=' * 55)
print('  AlphaGenome Predictions — Summary')
print('=' * 55)
print(f'  Total gene pairs loaded        : {total:,}')
print(f'  Complete (both expr computed)  : {complete:,}  ({100*complete/total:.1f}%)')
print(f'  Missing LFC                    : {null_lfc:,}')
print()
print(f'  LFC stats (complete pairs)')
print(f'    Mean        : {mean_lfc:.3f}')
print(f'    Median      : {median_lfc:.3f}')
print(f'    SD          : {std_lfc:.3f}')
print(f'    Human-biased (LFC>0) : {n_human_biased:,}  ({100*n_human_biased/len(lfc):.1f}%)')
print(f'    Chimp-biased (LFC<0) : {n_chimp_biased:,}  ({100*n_chimp_biased/len(lfc):.1f}%)')
print()
try:
    print(f'  vs Hybrid ASE (N={len(x):,} matched genes)')
    print(f'    Pearson r (LFC)      : {r_p:.3f}  p = {p_p:.2e}')
    print(f'    Spearman r (LFC)     : {r_s:.3f}  p = {p_s:.2e}')
    print(f'    Direction accuracy   : {100*n_agree/n_dir_total:.1f}%')
except NameError:
    print('  (hybrid comparison not run — check Section 6)')
print('=' * 55)